# 00. 06用データ取得: 米国債カーブ・Macro

このノートブックは、`06_simple_trend_cash_comparison.ipynb` を再現実行するために必要なデータをFREDから取得し、単一のExcelファイルに保存する。

出力ファイルは `data/us_treasury_macro_for_06.xlsx` である。06ノートブックは、このファイルだけを入力として使う。

## 設計方針

- データ取得元はFREDに統一する。
- 米国債カーブはCMT系列から作る。
- 主戦略 `MZ_SIMPLE_TREND_CASH` はBEIを使わないため、BEI/TIPSを必須条件にしない。
- 1981年から揃う `DGS3MO`, `DGS6MO`, `DGS1`, `DGS2`, `DGS3`, `DGS5`, `DGS7`, `DGS10`, `DGS30` を必須系列とする。
- `DGS20` は1987年から1993年に長期欠損があるため、必須ノードから外す。20年セクターは10年と30年の間の補間で作る。
- `T10YIE` と `DFII10` は2003年以降の任意系列として保存する。BEI系戦略は、BEIがない期間ではBEIペナルティを0として扱う。
- 未完了の当月は使わない。実行日の前月末を候補終端とし、必須系列が揃う最後の月末までを採用する。
- 日次系列の祝日欠損は直前値で補完する。ただし、補完前の欠損状況を必ず表で確認する。

この設計により、06の主戦略は1981年9月からの月次データを使える。24か月のローリング履歴を確保したうえで、最初の実現リターンは1983年10月末になる。

## 1. セットアップ

FRED APIキーは `.private/.env` の `FRED_API_KEY` から読む。

取得開始は1981年1月1日とする。Cash用の `DGS3MO` と短期カーブ用の `DGS6MO` は1981年9月から利用可能であり、これが主戦略を長期化するうえでの実質的な開始制約になる。

BEI `T10YIE` と10年TIPS実質金利 `DFII10` は2003年1月からしか利用できない。そのため、これらは任意系列として保存し、1981年からの主バックテストを制限しない。

In [1]:
import os
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from fredapi import Fred
from scipy.interpolate import CubicSpline

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_PATH = DATA_DIR / "us_treasury_macro_for_06.xlsx"

DATA_START = "1981-01-01"
RUN_DATE = pd.Timestamp.today().normalize()
TARGET_MONTH_END = (RUN_DATE.to_period("M") - 1).to_timestamp("M")

CURVE_NODES = np.array([0.5, 1, 2, 3, 5, 7, 10, 30], dtype=float)
MATURITIES = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 30], dtype=float)
SECTOR_NAMES = ["1Y", "2Y", "3Y", "4Y", "5Y", "6Y", "7Y", "8Y", "9Y", "10Y", "15Y", "20Y", "30Y"]

CURVE_SERIES = {
    0.5: "DGS6MO",
    1: "DGS1",
    2: "DGS2",
    3: "DGS3",
    5: "DGS5",
    7: "DGS7",
    10: "DGS10",
    30: "DGS30",
}

MANDATORY_MACRO_SERIES = ["DGS3MO", "DGS2", "DGS10", "DGS30"]
OPTIONAL_SERIES = ["DGS20", "T10YIE", "DFII10"]
MANDATORY_SERIES = list(dict.fromkeys(list(CURVE_SERIES.values()) + MANDATORY_MACRO_SERIES))
ALL_SERIES = list(dict.fromkeys(MANDATORY_SERIES + OPTIONAL_SERIES))

print(f"実行日: {RUN_DATE.date()}")
print(f"未完了の当月を避けるため、候補終端月末: {TARGET_MONTH_END.date()}")
print(f"取得開始日: {DATA_START}")
print(f"必須系列: {MANDATORY_SERIES}")
print(f"任意系列: {OPTIONAL_SERIES}")

実行日: 2026-07-12
未完了の当月を避けるため、候補終端月末: 2026-06-30
取得開始日: 1981-01-01
必須系列: ['DGS6MO', 'DGS1', 'DGS2', 'DGS3', 'DGS5', 'DGS7', 'DGS10', 'DGS30', 'DGS3MO']
任意系列: ['DGS20', 'T10YIE', 'DFII10']


## 2. FREDから日次系列を取得

必要系列は、必須系列と任意系列に分ける。

### 必須系列

カーブ構築と主戦略に必要な系列である。

- `DGS6MO`: 6か月CMT
- `DGS1`, `DGS2`, `DGS3`, `DGS5`, `DGS7`, `DGS10`, `DGS30`: 米国債CMT
- `DGS3MO`: Cash近似に使う3か月米国債利回り

### 任意系列

補助分析やBEI系戦略に使う系列である。これらの欠損は、主戦略のバックテスト期間を制限しない。

- `DGS20`: 20年CMT。ただし1987年から1993年に長期欠損があるため、カーブ補間の必須ノードにはしない。
- `T10YIE`: 10年BEI。2003年以降のみ利用可能。
- `DFII10`: 10年TIPS実質金利。2003年以降のみ利用可能。

In [2]:
def fetch_fred_series_with_retry(fred, series_id, start, end, max_attempts=5, sleep_seconds=2.0):
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            return fred.get_series(series_id, observation_start=start, observation_end=end)
        except Exception as exc:
            last_error = exc
            if attempt == max_attempts:
                raise
            wait = sleep_seconds * attempt
            print(f"{series_id}: FRED取得失敗 ({exc}); {wait:.1f}秒待って再試行します")
            time.sleep(wait)
    raise last_error

load_dotenv(dotenv_path="../.private/.env")
fred_api_key = os.environ["FRED_API_KEY"]
fred = Fred(api_key=fred_api_key)

raw = {}
for series_id in ALL_SERIES:
    s = fetch_fred_series_with_retry(
        fred,
        series_id,
        start=DATA_START,
        end=RUN_DATE.strftime("%Y-%m-%d"),
    )
    raw[series_id] = s
    valid = s.dropna()
    print(f"{series_id}: {valid.index.min().date()}〜{valid.index.max().date()}, obs={len(s)}, raw_nan={int(s.isna().sum())}")

raw_daily = pd.DataFrame(raw).sort_index()
raw_daily = raw_daily.loc[DATA_START:RUN_DATE.strftime("%Y-%m-%d"), ALL_SERIES]
raw_daily.tail()

DGS6MO: 1981-09-01〜2026-07-09, obs=11703, raw_nan=491


DGS1: 1981-01-02〜2026-07-09, obs=11876, raw_nan=496


DGS2: 1981-01-02〜2026-07-09, obs=11876, raw_nan=496


DGS3: 1981-01-02〜2026-07-09, obs=11876, raw_nan=496


DGS5: 1981-01-02〜2026-07-09, obs=11876, raw_nan=496


...

DGS3MO: 1981-09-01〜2026-07-09, obs=11703, raw_nan=491


DGS20: 1981-01-02〜2026-07-09, obs=11876, raw_nan=2185


T10YIE: 2003-01-02〜2026-07-10, obs=6137, raw_nan=253


DFII10: 2003-01-02〜2026-07-09, obs=6136, raw_nan=253


,DGS6MO,DGS1,DGS2,DGS3,DGS5,DGS7,DGS10,DGS30,DGS3MO,DGS20,T10YIE,DFII10
2026-07-06,3.98,3.95,4.13,4.14,4.21,4.33,4.48,4.99,3.87,4.99,2.24,2.24
2026-07-07,3.99,4.06,4.19,4.18,4.27,4.40,4.55,5.05,3.86,5.05,2.25,2.30
2026-07-08,3.99,4.06,4.21,4.21,4.31,4.43,4.56,5.06,3.87,5.07,2.25,2.31
2026-07-09,3.96,4.02,4.16,4.18,4.27,4.40,4.54,5.05,3.83,5.06,2.23,2.31
2026-07-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.24,NaN


## 3. 欠損確認と月末採用期間の決定

FREDの日次系列には、米国祝日などでNaNが入る。これは金利が欠落しているというより、その日に市場観測値がないという意味である。

ここでは以下を確認する。

1. 各系列の最初・最後の有効観測日
2. 最初の有効観測日以降の欠損数
3. 連続欠損の最大長
4. 月末化したあと、全系列が揃う月末範囲

連続欠損が極端に長い場合は、単純な直前値補完では危険である。このノートブックでは、各系列の連続欠損が5営業日相当以内であることを機械チェックする。

In [3]:
def max_consecutive_true(mask):
    mask = pd.Series(mask).fillna(False).astype(bool)
    if mask.empty:
        return 0
    groups = (mask != mask.shift()).cumsum()
    return int(mask.groupby(groups).sum().max())

coverage_rows = []
for series_id in ALL_SERIES:
    s = raw_daily[series_id]
    first_valid = s.first_valid_index()
    last_valid = s.last_valid_index()
    if first_valid is None:
        raw_nan_after_first = len(s)
        max_nan_run = len(s)
    else:
        active = s.loc[first_valid:last_valid]
        raw_nan_after_first = int(active.isna().sum())
        max_nan_run = max_consecutive_true(active.isna())
    coverage_rows.append({
        "series": series_id,
        "required_for_06_core": series_id in MANDATORY_SERIES,
        "first_valid": first_valid,
        "last_valid": last_valid,
        "observations": int(s.notna().sum()),
        "raw_nan_after_first": raw_nan_after_first,
        "max_consecutive_raw_nan": max_nan_run,
    })

coverage = pd.DataFrame(coverage_rows)
display(coverage)

mandatory_coverage = coverage[coverage["required_for_06_core"]]
if (mandatory_coverage["max_consecutive_raw_nan"] > 5).any():
    raise RuntimeError("必須系列に5営業日を超える連続欠損があります。ffillだけで処理してよいか確認してください。")

filled_daily = raw_daily.ffill()
monthly_all = filled_daily.resample("ME").last()
monthly_all = monthly_all.loc[:TARGET_MONTH_END]

complete_mask = monthly_all[MANDATORY_SERIES].notna().all(axis=1)
if not complete_mask.any():
    raise RuntimeError("必須系列が揃う月末がありません。")

FIRST_COMPLETE_MONTH_END = monthly_all.index[complete_mask][0]
LAST_COMPLETE_MONTH_END = monthly_all.index[complete_mask][-1]
monthly = monthly_all.loc[FIRST_COMPLETE_MONTH_END:LAST_COMPLETE_MONTH_END, ALL_SERIES]

monthly_missing_check = monthly_all.assign(
    mandatory_missing_count=monthly_all[MANDATORY_SERIES].isna().sum(axis=1),
    optional_missing_count=monthly_all[OPTIONAL_SERIES].isna().sum(axis=1),
    all_mandatory_complete=complete_mask,
)[["mandatory_missing_count", "optional_missing_count", "all_mandatory_complete"]]

print(f"最初の必須系列完備月末: {FIRST_COMPLETE_MONTH_END.date()}")
print(f"最後の必須系列完備月末: {LAST_COMPLETE_MONTH_END.date()}")
print(f"月次採用期間: {len(monthly)}か月")
print(f"必須系列の月次欠損セル数: {int(monthly[MANDATORY_SERIES].isna().sum().sum())}")
print(f"任意系列の月次欠損セル数: {int(monthly[OPTIONAL_SERIES].isna().sum().sum())}")

display(monthly_missing_check.head(12))
display(monthly_missing_check.tail(12))

assert monthly.index[-1] <= TARGET_MONTH_END
assert monthly[MANDATORY_SERIES].isna().sum().sum() == 0

,series,required_for_06_core,first_valid,last_valid,observations,raw_nan_after_first,max_consecutive_raw_nan
0,DGS6MO,True,1981-09-01,2026-07-09,11212,491,2
1,DGS1,True,1981-01-02,2026-07-09,11380,495,2
2,DGS2,True,1981-01-02,2026-07-09,11380,495,2
3,DGS3,True,1981-01-02,2026-07-09,11380,495,2
4,DGS5,True,1981-01-02,2026-07-09,11380,495,2
5,DGS7,True,1981-01-02,2026-07-09,11380,495,2
6,DGS10,True,1981-01-02,2026-07-09,11380,495,2
7,DGS30,True,1981-01-02,2026-07-09,11380,495,2
8,DGS3MO,True,1981-09-01,2026-07-09,11212,491,2
9,DGS20,False,1981-01-02,2026-07-09,9691,2184,1761


最初の必須系列完備月末: 1981-09-30
最後の必須系列完備月末: 2026-06-30
月次採用期間: 538か月
必須系列の月次欠損セル数: 0
任意系列の月次欠損セル数: 512


,mandatory_missing_count,optional_missing_count,all_mandatory_complete
1981-01-31,2,2,False
1981-02-28,2,2,False
1981-03-31,2,2,False
1981-04-30,2,2,False
1981-05-31,2,2,False
1981-06-30,2,2,False
1981-07-31,2,2,False
1981-08-31,2,2,False
1981-09-30,0,2,True
1981-10-31,0,2,True


,mandatory_missing_count,optional_missing_count,all_mandatory_complete
2025-07-31,0,0,True
2025-08-31,0,0,True
2025-09-30,0,0,True
2025-10-31,0,0,True
2025-11-30,0,0,True
2025-12-31,0,0,True
2026-01-31,0,0,True
2026-02-28,0,0,True
2026-03-31,0,0,True
2026-04-30,0,0,True


## 4. 13セクター利回りカーブを作る

06では、1Yから30Yまでの13セクターを使う。

補間元ノードは、6か月、1年、2年、3年、5年、7年、10年、30年である。

20年CMT `DGS20` はFRED上では利用可能な時期もあるが、1987年から1993年に長期欠損がある。したがって、長期バックテストでは必須ノードとして使わず、15年・20年セクターは10年と30年の間を含むカーブ補間で作る。

補間元ノードは `source_nodes`、補間後の13セクターは `yields` として保存する。

In [4]:
curve_source = monthly[list(CURVE_SERIES.values())].copy()
curve_source.columns = [f"{m:g}Y" for m in CURVE_NODES]

interpolated = []
for date, row in curve_source.iterrows():
    values = row.values.astype(float)
    if np.isnan(values).any():
        raise RuntimeError(f"{date.date()} のカーブノードに欠損があります。")
    cs = CubicSpline(CURVE_NODES, values, extrapolate=True)
    interpolated.append(cs(MATURITIES))

yields = pd.DataFrame(interpolated, index=curve_source.index, columns=SECTOR_NAMES)

print(f"source_nodes: {curve_source.index[0].date()}〜{curve_source.index[-1].date()}, shape={curve_source.shape}")
print(f"yields      : {yields.index[0].date()}〜{yields.index[-1].date()}, shape={yields.shape}")
print(f"欠損セル数  : source_nodes={int(curve_source.isna().sum().sum())}, yields={int(yields.isna().sum().sum())}")

display(curve_source.tail())
display(yields.tail())

assert curve_source.isna().sum().sum() == 0
assert yields.isna().sum().sum() == 0

source_nodes: 1981-09-30〜2026-06-30, shape=(538, 8)
yields      : 1981-09-30〜2026-06-30, shape=(538, 13)
欠損セル数  : source_nodes=0, yields=0


,0.5Y,1Y,2Y,3Y,5Y,7Y,10Y,30Y
2026-02-28,3.60,3.48,3.38,3.39,3.51,3.72,3.97,4.64
2026-03-31,3.72,3.68,3.79,3.81,3.92,4.11,4.30,4.88
2026-04-30,3.71,3.72,3.88,3.91,4.02,4.20,4.40,4.98
2026-05-31,3.78,3.79,3.98,4.06,4.13,4.27,4.45,4.99
2026-06-30,4.01,3.98,4.14,4.15,4.19,4.30,4.44,4.91


,1Y,2Y,3Y,4Y,5Y,6Y,7Y,8Y,9Y,10Y,15Y,20Y,30Y
2026-02-28,3.48,3.38,3.39,3.436636,3.51,3.610859,3.72,3.817474,3.900362,3.97,4.166259,4.231894,4.64
2026-03-31,3.68,3.79,3.81,3.843351,3.92,4.015303,4.11,4.189463,4.252198,4.30,4.377724,4.350938,4.88
2026-04-30,3.72,3.88,3.91,3.944883,4.02,4.110213,4.20,4.278629,4.344901,4.40,4.549367,4.597107,4.98
2026-05-31,3.79,3.98,4.06,4.090746,4.13,4.194375,4.27,4.339107,4.398824,4.45,4.607548,4.679180,4.99
2026-06-30,3.98,4.14,4.15,4.151106,4.19,4.243544,4.30,4.352035,4.398549,4.44,4.587308,4.676630,4.91


## 5. Macro系列を作る

06で使うMacro系列は以下である。

- `DGS3MO`: Cashリターン近似
- `DGS2`, `DGS10`: 名目金利トレンド、2s10反転判定
- `DGS30`: 長期金利確認用
- `T10YIE`: BEI Momentum
- `DFII10`: 10年TIPS実質金利
- `REAL10_PROXY = DGS10 - T10YIE`: 近似実質金利
- `SLOPE_10Y_3M = DGS10 - DGS3MO`: 10年対3か月スロープ

`REAL10_PROXY` は参考列であり、06の主戦略の意思決定には直接使わない。

In [5]:
macro_monthly = monthly[["DGS3MO", "DGS2", "DGS10", "DGS30", "T10YIE", "DFII10"]].copy()
macro_monthly["REAL10_PROXY"] = macro_monthly["DGS10"] - macro_monthly["T10YIE"]
macro_monthly["SLOPE_10Y_3M"] = macro_monthly["DGS10"] - macro_monthly["DGS3MO"]
macro_monthly = macro_monthly[
    ["DGS3MO", "DGS2", "DGS10", "DGS30", "T10YIE", "DFII10", "REAL10_PROXY", "SLOPE_10Y_3M"]
]

macro_daily = filled_daily.loc[:LAST_COMPLETE_MONTH_END, ["DGS3MO", "DGS2", "DGS10", "DGS30", "T10YIE", "DFII10"]].copy()

mandatory_macro_cols = ["DGS3MO", "DGS2", "DGS10", "DGS30", "SLOPE_10Y_3M"]
optional_macro_cols = ["T10YIE", "DFII10", "REAL10_PROXY"]

print(f"macro_monthly: {macro_monthly.index[0].date()}〜{macro_monthly.index[-1].date()}, shape={macro_monthly.shape}")
print(f"macro_daily  : {macro_daily.index[0].date()}〜{macro_daily.index[-1].date()}, shape={macro_daily.shape}")
print(f"必須Macro欠損: monthly={int(macro_monthly[mandatory_macro_cols].isna().sum().sum())}")
print(f"任意Macro欠損: monthly={int(macro_monthly[optional_macro_cols].isna().sum().sum())}")
print(f"任意Macroの最初の有効月:")
print(macro_monthly[optional_macro_cols].apply(lambda s: s.first_valid_index()).to_string())

display(macro_monthly.head())
display(macro_monthly.tail())

assert macro_monthly[mandatory_macro_cols].isna().sum().sum() == 0
# dailyの初期日には、系列開始前または祝日由来のNaNが残りうる。06はmonthlyだけを使う。

macro_monthly: 1981-09-30〜2026-06-30, shape=(538, 8)
macro_daily  : 1981-01-01〜2026-06-30, shape=(11869, 6)
必須Macro欠損: monthly=0
任意Macro欠損: monthly=768
任意Macroの最初の有効月:
T10YIE         2003-01-31
DFII10         2003-01-31
REAL10_PROXY   2003-01-31


,DGS3MO,DGS2,DGS10,DGS30,T10YIE,DFII10,REAL10_PROXY,SLOPE_10Y_3M
1981-09-30,15.05,16.69,15.84,15.19,NaN,NaN,NaN,0.79
1981-10-31,13.33,14.59,14.63,14.36,NaN,NaN,NaN,1.30
1981-11-30,10.78,12.36,13.13,12.91,NaN,NaN,NaN,2.35
1981-12-31,11.54,13.63,13.98,13.65,NaN,NaN,NaN,2.44
1982-01-31,13.08,14.24,14.14,13.91,NaN,NaN,NaN,1.06


,DGS3MO,DGS2,DGS10,DGS30,T10YIE,DFII10,REAL10_PROXY,SLOPE_10Y_3M
2026-02-28,3.67,3.38,3.97,4.64,2.25,1.72,1.72,0.30
2026-03-31,3.70,3.79,4.30,4.88,2.30,2.00,2.00,0.60
2026-04-30,3.68,3.88,4.40,4.98,2.46,1.94,1.94,0.72
2026-05-31,3.69,3.98,4.45,4.99,2.38,2.07,2.07,0.76
2026-06-30,3.87,4.14,4.44,4.91,2.24,2.20,2.20,0.57


## 6. 06用のバックテスト可能期間を確認する

06は、各シグナル日に過去24か月の利回りカーブを使って因子モデルを推定し、翌月の実現リターンを評価する。

したがって、データ期間が $[t_0, t_N]$ のとき、最初のシグナル日は24か月分の履歴を持つ $t_{24}$、最後のシグナル日は翌月リターンが観測できる $t_{N-1}$ である。

このノートブックでは、06が自動的にこの範囲を使えるように、期間情報も `metadata` シートへ保存する。

In [6]:
LOOKBACK_MONTHS = 24
first_signal_date = yields.index[LOOKBACK_MONTHS]
first_return_date = yields.index[LOOKBACK_MONTHS + 1]
last_signal_date = yields.index[-2]
last_return_date = yields.index[-1]

metadata = pd.DataFrame([
    {"key": "run_date", "value": RUN_DATE.date().isoformat()},
    {"key": "target_month_end", "value": TARGET_MONTH_END.date().isoformat()},
    {"key": "first_complete_month_end", "value": FIRST_COMPLETE_MONTH_END.date().isoformat()},
    {"key": "last_complete_month_end", "value": LAST_COMPLETE_MONTH_END.date().isoformat()},
    {"key": "lookback_months", "value": LOOKBACK_MONTHS},
    {"key": "first_signal_date", "value": first_signal_date.date().isoformat()},
    {"key": "first_return_date", "value": first_return_date.date().isoformat()},
    {"key": "last_signal_date", "value": last_signal_date.date().isoformat()},
    {"key": "last_return_date", "value": last_return_date.date().isoformat()},
])

print(f"06で利用可能な最初のシグナル日: {first_signal_date.date()}")
print(f"06で利用可能な最初のリターン日: {first_return_date.date()}")
print(f"06で利用可能な最後のシグナル日: {last_signal_date.date()}")
print(f"06で利用可能な最後のリターン日: {last_return_date.date()}")

display(metadata)

06で利用可能な最初のシグナル日: 1983-09-30
06で利用可能な最初のリターン日: 1983-10-31
06で利用可能な最後のシグナル日: 2026-05-31
06で利用可能な最後のリターン日: 2026-06-30


,key,value
0,run_date,2026-07-12
1,target_month_end,2026-06-30
2,first_complete_month_end,1981-09-30
3,last_complete_month_end,2026-06-30
4,lookback_months,24
5,first_signal_date,1983-09-30
6,first_return_date,1983-10-31
7,last_signal_date,2026-05-31
8,last_return_date,2026-06-30


## 7. Excelへ保存

06はこのExcelだけを読む。

保存するシートは以下である。

- `yields`: 13セクター利回り
- `source_nodes`: 補間元のCMTノード。`DGS20` は長期欠損があるため含めない
- `macro_monthly`: 06で使う月次Macro系列。BEI/TIPSは2003年以前に欠損を許容する任意列
- `macro_daily`: Macro系列の日次補完済みデータ
- `raw_daily`: FREDから取得した未補完の日次データ
- `coverage`: 系列ごとの取得範囲・欠損確認
- `monthly_missing_check`: 月末ごとの必須・任意系列の欠損確認
- `metadata`: 06のバックテスト可能期間など

In [7]:
with pd.ExcelWriter(OUTPUT_PATH) as writer:
    yields.to_excel(writer, sheet_name="yields")
    curve_source.to_excel(writer, sheet_name="source_nodes")
    macro_monthly.to_excel(writer, sheet_name="macro_monthly")
    macro_daily.to_excel(writer, sheet_name="macro_daily")
    raw_daily.to_excel(writer, sheet_name="raw_daily")
    coverage.to_excel(writer, sheet_name="coverage", index=False)
    monthly_missing_check.to_excel(writer, sheet_name="monthly_missing_check")
    metadata.to_excel(writer, sheet_name="metadata", index=False)

print(f"保存完了: {OUTPUT_PATH}")
print(f"06入力データ期間: {yields.index[0].date()}〜{yields.index[-1].date()}")
print(f"06バックテスト想定リターン期間: {first_return_date.date()}〜{last_return_date.date()}")

保存完了: ../data/us_treasury_macro_for_06.xlsx
06入力データ期間: 1981-09-30〜2026-06-30
06バックテスト想定リターン期間: 1983-10-31〜2026-06-30
